# Detection v3 — YOLOv8x + Enhanced Augmentation + SAHI + Full Crash Recovery

## Improvements over v2-retrain (55.22% test mAP@50)

| | v2-retrain | v3 |
|---|---|---|
| Model | YOLOv8l | **YOLOv8x** (+15% params) |
| weight_decay | 0.0005 | **0.001** |
| label_smoothing | — | **0.05** |
| copy_paste | — | **0.3** |
| perspective | — | **0.0002** |
| close_mosaic | 10 | **15** |
| Phase 2 epochs | 70 | **80** |
| TTA at inference | ✗ | **✓** |
| SAHI evaluation | ✗ | **✓** |
| Mid-epoch Drive backup | ✗ | **every 10 ep** |

**Drive saves to `checkpoints/detector_v3/` — all previous runs untouched.**

Expected: 58–65% test mAP@50

## Setup & Dependencies

In [1]:
import subprocess
import sys

packages = [
    'ultralytics>=8.0.0',
    'sahi>=0.11.0',
    'torch',
    'torchvision',
    'opencv-python',
    'scikit-learn',
    'matplotlib',
    'pandas',
    'seaborn',
    'tqdm',
    'pyyaml',
]

print('Installing dependencies...')
for pkg in packages:
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
    except Exception as e:
        print(f'  Warning: {pkg} — {e}')
print('Done.')

Installing dependencies...
Done.


## Imports & Environment

In [2]:
import os
import sys
import json
import shutil
import random
import zipfile
from pathlib import Path
from datetime import datetime

import cv2
import numpy as np
import yaml
import torch
import matplotlib.pyplot as plt
import pandas as pd
from tqdm.auto import tqdm
from ultralytics import YOLO

# ── Environment ───────────────────────────────────────────────
try:
    from google.colab import drive
    IN_COLAB = True
    print('Colab detected — mounting Drive...')
    drive.mount('/content/drive', force_remount=False)
    DRIVE_DIR = Path('/content/drive/MyDrive/HeritagePreservation')
    DATA_DIR  = Path('/content/data')
except ImportError:
    IN_COLAB = False
    print('Running locally')
    notebook_dir = Path.cwd()
    project_root = notebook_dir.parent if notebook_dir.name == 'notebooks' else notebook_dir
    DATA_DIR  = project_root / 'data'
    DRIVE_DIR = project_root

# ── v3-specific dirs (previous runs untouched) ────────────────
CHECKPOINT_DIR = (
    Path('/content/checkpoints/detector_v3') if IN_COLAB
    else DATA_DIR.parent / 'checkpoints' / 'detector_v3'
)
DRIVE_CKPT = DRIVE_DIR / 'checkpoints' / 'detector_v3'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f'Device         : {device}')
print(f'DATA_DIR       : {DATA_DIR}')
print(f'CHECKPOINT_DIR : {CHECKPOINT_DIR}')
print(f'DRIVE_CKPT     : {DRIVE_CKPT}')
print(f'IN_COLAB       : {IN_COLAB}')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Colab detected — mounting Drive...
Mounted at /content/drive
Device         : cuda:0
DATA_DIR       : /content/data
CHECKPOINT_DIR : /content/checkpoints/detector_v3
DRIVE_CKPT     : /content/drive/MyDrive/HeritagePreservation/checkpoints/detector_v3
IN_COLAB       : True


## Drive Backup / Restore Helpers

In [3]:
def backup_to_drive(local_path, drive_name):
    """Copy local file to Drive checkpoint dir."""
    local_path = Path(local_path)
    if IN_COLAB and local_path.exists():
        DRIVE_CKPT.mkdir(parents=True, exist_ok=True)
        dst = DRIVE_CKPT / drive_name
        shutil.copy2(local_path, dst)
        mb = local_path.stat().st_size / 1e6
        print(f'  Drive backup: {drive_name} ({mb:.1f} MB)')

def restore_from_drive(drive_name, local_path):
    """Restore from Drive if local file is missing."""
    local_path = Path(local_path)
    if IN_COLAB and not local_path.exists():
        src = DRIVE_CKPT / drive_name
        if src.exists():
            local_path.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, local_path)
            print(f'  Restored from Drive: {drive_name}')
            return True
    return False

## Data Loading (OmniCrack Only)

In [4]:
import subprocess
r = subprocess.run(['ls', '-la', str(DATA_DIR)], capture_output=True, text=True)
print('DATA_DIR contents:')
print(r.stdout or r.stderr or 'empty / not found')

DATA_DIR contents:
ls: cannot access '/content/data': No such file or directory



In [6]:
DATA_DIR.mkdir(parents=True, exist_ok=True)

def extract_dataset(zip_name, data_dir, drive_dir):
    """Copy zip from Drive to local /content, then extract.
    Retries up to 3x with force_remount on OSError (FUSE disconnect on large files).
    """
    zip_path = drive_dir / zip_name
    out_name = zip_name.replace('.zip', '')
    out_dir  = data_dir / out_name
    if not zip_path.exists():
        print(f'SKIP {zip_name} — not on Drive')
        return
    if out_dir.exists() and any(out_dir.rglob('*.*')):
        print(f'{zip_name}: already extracted')
        return
    size_mb   = zip_path.stat().st_size / 1e6
    local_zip = Path(f'/content/_tmp_{out_name}.zip')
    print(f'Copying {zip_name} ({size_mb:.0f} MB) to local...')
    for attempt in range(3):
        try:
            if local_zip.exists():
                local_zip.unlink()
            shutil.copy2(zip_path, local_zip)
            break
        except OSError as e:
            if local_zip.exists():
                local_zip.unlink()
            if attempt < 2:
                print(f'  Copy failed (attempt {attempt + 1}/3): {e}')
                print('  Remounting Drive...')
                from google.colab import drive as _drive
                _drive.mount('/content/drive', force_remount=True)
            else:
                raise
    print('Extracting...')
    with zipfile.ZipFile(local_zip, 'r') as zf:
        zf.extractall(data_dir)
    local_zip.unlink()
    print('Done.')

if IN_COLAB:
    extract_dataset('omnicrack30k.zip', DATA_DIR, DRIVE_DIR)
else:
    print('Local mode — skipping Drive extraction')

print('\n' + '='*60)

# ── Locate OmniCrack structure ────────────────────────────────
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
IMG_ROOT = ANN_ROOT = OMNI_DIR = None

for candidate in [DATA_DIR / 'omnicrack30k', DATA_DIR]:
    if (candidate / 'images').exists() and (candidate / 'annotations').exists():
        IMG_ROOT = candidate / 'images'
        ANN_ROOT = candidate / 'annotations'
        OMNI_DIR = candidate
        print(f'OmniCrack found: {OMNI_DIR}')
        break

if OMNI_DIR is None:
    print('ERROR: OmniCrack not found')
    print(f'  Checked: {DATA_DIR / "omnicrack30k"}')
    print(f'  Checked: {DATA_DIR}')
    raise FileNotFoundError(f'No OmniCrack data in {DATA_DIR}')

SPLITS = ['training', 'validation', 'test']

# Build mask lookup: stem -> path
mask_lookup = {}
for split in SPLITS:
    split_ann_dir = ANN_ROOT / split
    if split_ann_dir.exists():
        for mask_path in split_ann_dir.rglob('*.*'):
            if mask_path.suffix.lower() in IMG_EXTS:
                mask_lookup[mask_path.stem] = mask_path

# Collect image-mask pairs
omnicrack_samples = []
for split in SPLITS:
    split_img_dir = IMG_ROOT / split
    if not split_img_dir.exists():
        print(f'  Split {split}: not found')
        continue
    for img_path in split_img_dir.rglob('*.*'):
        if img_path.suffix.lower() not in IMG_EXTS:
            continue
        mask = mask_lookup.get(img_path.stem)
        if mask:
            omnicrack_samples.append((str(img_path), str(mask)))

print(f'OmniCrack samples found: {len(omnicrack_samples)}')
if not omnicrack_samples:
    raise FileNotFoundError('No image-mask pairs found')

# Limit to 5000 for iteration speed (same as v2-retrain)
if len(omnicrack_samples) > 5000:
    rng = random.Random(42)
    rng.shuffle(omnicrack_samples)
    omnicrack_samples = omnicrack_samples[:5000]
    print(f'Limited to: {len(omnicrack_samples)} (speed)')

Copying omnicrack30k.zip (10604 MB) to local...
Extracting...
Done.

OmniCrack found: /content/data
OmniCrack samples found: 30017
Limited to: 5000 (speed)


## Convert Masks to YOLO Labels

In [7]:
def mask_to_yolo_labels(mask_path, min_area=200, max_area_frac=0.40, max_ar=8.0):
    """Convert binary mask to YOLO bbox labels.
    Filters: spanning boxes (>40% image), extreme aspect ratio (>8:1), tiny blobs.
    """
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return []
    h, w = mask.shape
    _, binary = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    labels = []
    for cnt in contours:
        if cv2.contourArea(cnt) < min_area:
            continue
        x, y, bw, bh = cv2.boundingRect(cnt)
        bw_n = bw / w
        bh_n = bh / h
        if bw_n * bh_n > max_area_frac:
            continue
        if max(bw_n / (bh_n + 1e-6), bh_n / (bw_n + 1e-6)) > max_ar:
            continue
        cx = (x + bw / 2) / w
        cy = (y + bh / 2) / h
        labels.append(f'0 {cx:.6f} {cy:.6f} {min(bw_n, 1.):.6f} {min(bh_n, 1.):.6f}')
    return labels

labels_dir = OMNI_DIR / 'labels'
labels_dir.mkdir(parents=True, exist_ok=True)

print('Converting masks to YOLO labels...')
total_boxes = total_imgs = 0
for img_path, mask_path in omnicrack_samples:
    labels = mask_to_yolo_labels(mask_path)
    if not labels:
        continue
    img_obj = Path(img_path)
    lbl_out = labels_dir / img_obj.relative_to(IMG_ROOT).with_suffix('.txt')
    lbl_out.parent.mkdir(parents=True, exist_ok=True)
    lbl_out.write_text('\n'.join(labels))
    total_boxes += len(labels)
    total_imgs  += 1

print(f'Converted: {total_imgs} images, {total_boxes} boxes')

# Re-collect only images that have valid labels
omnicrack_samples_filtered = []
for split in SPLITS:
    split_img_dir = IMG_ROOT / split
    if not split_img_dir.exists():
        continue
    for img_path in split_img_dir.rglob('*.*'):
        if img_path.suffix.lower() not in IMG_EXTS:
            continue
        lbl = labels_dir / split / img_path.relative_to(split_img_dir).with_suffix('.txt')
        if lbl.exists() and lbl.stat().st_size > 5:
            omnicrack_samples_filtered.append((str(img_path), str(lbl)))

print(f'Valid samples (with labels): {len(omnicrack_samples_filtered)}')
omnicrack_samples = omnicrack_samples_filtered

Converting masks to YOLO labels...
Converted: 612 images, 951 boxes
Valid samples (with labels): 612


## Create Dataset Splits & YAML

In [8]:
from sklearn.model_selection import train_test_split

train_samples, temp  = train_test_split(omnicrack_samples, test_size=0.30, random_state=42)
val_samples, test_samples = train_test_split(temp, test_size=0.50, random_state=42)
print(f'Train: {len(train_samples)} | Val: {len(val_samples)} | Test: {len(test_samples)}')

TRAIN_DIR = CHECKPOINT_DIR / 'train'
VAL_DIR   = CHECKPOINT_DIR / 'val'
TEST_DIR  = CHECKPOINT_DIR / 'test'

for d in [TRAIN_DIR, VAL_DIR, TEST_DIR]:
    (d / 'images').mkdir(parents=True, exist_ok=True)
    (d / 'labels').mkdir(parents=True, exist_ok=True)

def setup_split(samples, split_dir):
    for img_path, label_path in samples:
        img_name   = Path(img_path).name
        label_name = Path(label_path).name
        dst_img = split_dir / 'images' / img_name
        if not dst_img.exists():
            try:
                os.symlink(img_path, dst_img)
            except Exception:
                shutil.copy2(img_path, dst_img)
        dst_lbl = split_dir / 'labels' / label_name
        if not dst_lbl.exists():
            shutil.copy2(label_path, dst_lbl)

print('Setting up splits...')
for samples, split_dir, name in [
    (train_samples, TRAIN_DIR, 'train'),
    (val_samples,   VAL_DIR,   'val'),
    (test_samples,  TEST_DIR,  'test'),
]:
    setup_split(samples, split_dir)
    print(f'  {name}: {len(list((split_dir / "images").glob("*.*")))} images')

dataset_config = {
    'path':  str(CHECKPOINT_DIR),
    'train': 'train',
    'val':   'val',
    'test':  'test',
    'nc':    1,
    'names': {0: 'crack'},
}
yaml_path = CHECKPOINT_DIR / 'dataset.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(dataset_config, f)
print(f'\nDataset YAML: {yaml_path}')

Train: 428 | Val: 92 | Test: 92
Setting up splits...
  train: 428 images
  val: 92 images
  test: 92 images

Dataset YAML: /content/checkpoints/detector_v3/dataset.yaml


## Training History & Crash Recovery

In [9]:
HIST_DEFAULT = {
    'p1_done': False, 'p2_done': False, 'p3_done': False,
    'p1_map':  0.0,   'p2_map':  0.0,   'p3_map':  0.0,
}
hist_file = CHECKPOINT_DIR / 'history.json'

# Load: local first, Drive fallback (for post-VM-recycle resume)
if hist_file.exists():
    hist = json.load(open(hist_file))
    print('History loaded (local)')
elif IN_COLAB and (DRIVE_CKPT / 'history.json').exists():
    hist = json.load(open(DRIVE_CKPT / 'history.json'))
    shutil.copy2(DRIVE_CKPT / 'history.json', hist_file)
    print('History restored from Drive')
else:
    hist = dict(HIST_DEFAULT)
    print('Fresh start')

for k, v in HIST_DEFAULT.items():
    hist.setdefault(k, v)

print(f'  P1: done={hist["p1_done"]}  mAP={hist["p1_map"]:.4f}')
print(f'  P2: done={hist["p2_done"]}  mAP={hist["p2_map"]:.4f}')
print(f'  P3: done={hist["p3_done"]}  mAP={hist["p3_map"]:.4f}')

def save_hist():
    json.dump(hist, open(hist_file, 'w'), indent=2)
    backup_to_drive(hist_file, 'history.json')

Fresh start
  P1: done=False  mAP=0.0000
  P2: done=False  mAP=0.0000
  P3: done=False  mAP=0.0000


## Phase 1: Low Resolution (320px, 20 epochs)

YOLOv8x warm-up at 320px. Backbone learns crack features before high-res fine-tune.

In [10]:
P1_DIR  = CHECKPOINT_DIR / 'phase1_320px'
P1_BEST = P1_DIR / 'weights' / 'best.pt'
P1_LAST = P1_DIR / 'weights' / 'last.pt'

if not hist['p1_done']:
    print('\n=== PHASE 1: 320px — 20 epochs (YOLOv8x) ===')

    # Try Drive restore for mid-epoch resume
    restore_from_drive('phase1_last.pt', P1_LAST)

    if P1_LAST.exists():
        print('Resuming Phase 1 from last.pt...')
        m = YOLO(str(P1_LAST))
        m.train(resume=True)
    else:
        m = YOLO('yolov8x.pt')
        m.train(
            data=str(yaml_path), imgsz=320, epochs=20, batch=16,
            device=device,
            patience=10, save=True, save_period=5,
            project=str(CHECKPOINT_DIR), name='phase1_320px', exist_ok=True,
            lr0=0.001, lrf=0.1, warmup_epochs=10, warmup_momentum=0.8,
            momentum=0.937, weight_decay=0.001,
            cos_lr=True,
            hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
            degrees=15, translate=0.1, scale=0.5,
            flipud=0.5, fliplr=0.5, mosaic=1.0,
            val=True, verbose=True,
        )

    if P1_BEST.exists():
        v = YOLO(str(P1_BEST)).val(data=str(yaml_path), split='val', verbose=False)
        hist['p1_map'] = float(v.box.map50)
        print(f'Phase 1 mAP@50 (val): {hist["p1_map"]:.4f}')

    hist['p1_done'] = True
    save_hist()
    backup_to_drive(P1_BEST, 'phase1_best.pt')
    backup_to_drive(P1_LAST, 'phase1_last.pt')
else:
    print(f'Phase 1 complete — mAP@50={hist["p1_map"]:.4f}')


=== PHASE 1: 320px — 20 epochs (YOLOv8x) ===
Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/checkpoints/detector_v3/dataset.yaml, degrees=15, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.1, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8x.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=phase1_320px, nbs=64, nms=False, opset=None, o

## Phase 2: Medium Resolution (640px, 80 epochs)

Key additions over v2-retrain:
- `label_smoothing=0.05` — reduces overconfidence on val set
- `weight_decay=0.001` — stronger regularisation
- `copy_paste=0.3` — pastes crack instances across images
- `perspective=0.0002` — geometric diversity
- `close_mosaic=15` — keeps mosaic active longer
- Drive backup every 10 epochs via callback

In [11]:
P2_DIR  = CHECKPOINT_DIR / 'phase2_640px'
P2_BEST = P2_DIR / 'weights' / 'best.pt'
P2_LAST = P2_DIR / 'weights' / 'last.pt'

if not hist['p2_done']:
    print('\n=== PHASE 2: 640px — 80 epochs ===')

    # Restore checkpoints if VM was recycled
    restore_from_drive('phase2_last.pt', P2_LAST)
    restore_from_drive('phase1_best.pt', P1_BEST)

    # Callback: back up last.pt to Drive every 10 epochs
    def _backup_p2(trainer):
        if trainer.epoch > 0 and trainer.epoch % 10 == 0:
            last = Path(trainer.save_dir) / 'weights' / 'last.pt'
            backup_to_drive(last, 'phase2_last.pt')

    if P2_LAST.exists():
        print('Resuming Phase 2 from last.pt...')
        m = YOLO(str(P2_LAST))
        m.add_callback('on_train_epoch_end', _backup_p2)
        m.train(resume=True)
    else:
        if not P1_BEST.exists():
            raise FileNotFoundError(f'Phase 1 checkpoint not found: {P1_BEST}')
        m = YOLO(str(P1_BEST))
        m.add_callback('on_train_epoch_end', _backup_p2)
        m.train(
            data=str(yaml_path), imgsz=640, epochs=80, batch=16,
            device=device,
            patience=25, save=True, save_period=10,
            project=str(CHECKPOINT_DIR), name='phase2_640px', exist_ok=True,
            lr0=0.0005, lrf=0.0001, warmup_epochs=5, warmup_momentum=0.8,
            momentum=0.937, weight_decay=0.001,
            label_smoothing=0.05,
            cos_lr=True,
            hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
            degrees=15, translate=0.15, scale=0.5,
            perspective=0.0002,
            flipud=0.5, fliplr=0.5,
            mosaic=1.0, mixup=0.2, copy_paste=0.3,
            close_mosaic=15,
            val=True, verbose=True,
        )

    if P2_BEST.exists():
        v = YOLO(str(P2_BEST)).val(data=str(yaml_path), split='val', verbose=False)
        hist['p2_map'] = float(v.box.map50)
        print(f'Phase 2 mAP@50 (val): {hist["p2_map"]:.4f}')

    hist['p2_done'] = True
    save_hist()
    backup_to_drive(P2_BEST, 'phase2_best.pt')
    backup_to_drive(P2_LAST, 'phase2_last.pt')
else:
    print(f'Phase 2 complete — mAP@50={hist["p2_map"]:.4f}')


=== PHASE 2: 640px — 80 epochs ===
WARNING ⚠️ 'label_smoothing' is deprecated and will be removed in the future.
Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=15, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/checkpoints/detector_v3/dataset.yaml, degrees=15, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.0001, mask_ratio=4, max_det=300, mixup=0.2, mode=train, model=/content/checkpoints/detector_v3/pha

## Phase 3: High Resolution (1024px, 30 epochs)

Fine-tune at 1024px for thin crack detail. `batch=8` to avoid OOM with YOLOv8x.

In [12]:
P3_DIR  = CHECKPOINT_DIR / 'phase3_1024px'
P3_BEST = P3_DIR / 'weights' / 'best.pt'
P3_LAST = P3_DIR / 'weights' / 'last.pt'

RUN_P3 = True  # set False to skip Phase 3

if RUN_P3 and not hist['p3_done']:
    print('\n=== PHASE 3: 1024px — 30 epochs (batch=8) ===')

    restore_from_drive('phase3_last.pt', P3_LAST)
    restore_from_drive('phase2_best.pt', P2_BEST)

    def _backup_p3(trainer):
        if trainer.epoch > 0 and trainer.epoch % 5 == 0:
            last = Path(trainer.save_dir) / 'weights' / 'last.pt'
            backup_to_drive(last, 'phase3_last.pt')

    if P3_LAST.exists():
        print('Resuming Phase 3 from last.pt...')
        m = YOLO(str(P3_LAST))
        m.add_callback('on_train_epoch_end', _backup_p3)
        m.train(resume=True)
    else:
        if not P2_BEST.exists():
            raise FileNotFoundError(f'Phase 2 checkpoint not found: {P2_BEST}')
        m = YOLO(str(P2_BEST))
        m.add_callback('on_train_epoch_end', _backup_p3)
        m.train(
            data=str(yaml_path), imgsz=1024, epochs=30, batch=8,
            device=device,
            patience=15, save=True, save_period=5,
            project=str(CHECKPOINT_DIR), name='phase3_1024px', exist_ok=True,
            lr0=0.0002, lrf=0.00001, warmup_epochs=3, warmup_momentum=0.8,
            momentum=0.937, weight_decay=0.001,
            label_smoothing=0.05,
            cos_lr=True,
            hsv_h=0.01, hsv_s=0.5, hsv_v=0.3,
            degrees=10, translate=0.1, scale=0.4,
            flipud=0.3, fliplr=0.5,
            mosaic=0.9, mixup=0.1, copy_paste=0.1,
            close_mosaic=10,
            val=True, verbose=True,
        )

    if P3_BEST.exists():
        v = YOLO(str(P3_BEST)).val(data=str(yaml_path), split='val', verbose=False)
        hist['p3_map'] = float(v.box.map50)
        print(f'Phase 3 mAP@50 (val): {hist["p3_map"]:.4f}')

    hist['p3_done'] = True
    save_hist()
    backup_to_drive(P3_BEST, 'phase3_best.pt')
    backup_to_drive(P3_LAST, 'phase3_last.pt')

elif hist['p3_done']:
    print(f'Phase 3 complete — mAP@50={hist["p3_map"]:.4f}')
else:
    print('Phase 3 disabled (RUN_P3=False)')


=== PHASE 3: 1024px — 30 epochs (batch=8) ===
WARNING ⚠️ 'label_smoothing' is deprecated and will be removed in the future.
Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/checkpoints/detector_v3/dataset.yaml, degrees=10, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.3, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.01, hsv_s=0.5, hsv_v=0.3, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0002, lrf=1e-05, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=/content/checkpoints/detect

## Best Model Selection

In [13]:
candidates = [
    ('P1 (320px)',  hist['p1_map'], CHECKPOINT_DIR / 'phase1_320px'  / 'weights' / 'best.pt', hist['p1_done']),
    ('P2 (640px)',  hist['p2_map'], CHECKPOINT_DIR / 'phase2_640px'  / 'weights' / 'best.pt', hist['p2_done']),
    ('P3 (1024px)', hist['p3_map'], CHECKPOINT_DIR / 'phase3_1024px' / 'weights' / 'best.pt', hist['p3_done']),
]

valid = [(name, score, path) for name, score, path, done in candidates
         if done and path.exists()]

if not valid:
    raise RuntimeError('No completed phase checkpoint found. Run training phases first.')

best_phase_name, best_phase_map, best_ckpt = max(valid, key=lambda x: x[1])

print('Phase results (val mAP@50):')
for name, score, path, done in candidates:
    status = f'{score:.4f}' if done else 'not run'
    marker = ' <-- BEST' if (done and path == best_ckpt) else ''
    print(f'  {name:12}: {status}{marker}')
print(f'\nBest checkpoint: {best_ckpt}')

Phase results (val mAP@50):
  P1 (320px)  : 0.4242
  P2 (640px)  : 0.5073 <-- BEST
  P3 (1024px) : 0.4882

Best checkpoint: /content/checkpoints/detector_v3/phase2_640px/weights/best.pt


## Evaluation: Standard + TTA + Confidence Sweep

TTA (`augment=True`) runs flipped/scaled versions at inference and merges predictions.

In [14]:
print('=== INFERENCE COMPARISON ===')
m_eval = YOLO(str(best_ckpt))

# Standard inference
print('Standard inference...')
std_v    = m_eval.val(data=str(yaml_path), split='test', device=device, verbose=False)
std_map50 = float(std_v.box.map50)
std_prec  = float(std_v.box.mp)
std_rec   = float(std_v.box.mr)
print(f'Standard mAP@50 : {std_map50:.4f}  P={std_prec:.3f}  R={std_rec:.3f}')

# TTA inference
print('TTA inference (augment=True)...')
tta_v    = m_eval.val(data=str(yaml_path), split='test', device=device,
                      augment=True, verbose=False)
tta_map50 = float(tta_v.box.map50)
tta_prec  = float(tta_v.box.mp)
tta_rec   = float(tta_v.box.mr)
print(f'TTA mAP@50      : {tta_map50:.4f}  P={tta_prec:.3f}  R={tta_rec:.3f}')

# Confidence threshold sweep
print('\nConfidence sweep...')
best_conf, best_map = 0.25, 0.0
conf_results = {}
for conf in [0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]:
    v = m_eval.val(data=str(yaml_path), split='test', device=device,
                   conf=conf, iou=0.5, verbose=False)
    map50 = float(v.box.map50)
    conf_results[conf] = {
        'map50': map50,
        'precision': float(v.box.mp),
        'recall':    float(v.box.mr),
    }
    print(f'  conf={conf:.2f}: mAP@50={map50:.4f}  P={float(v.box.mp):.3f}  R={float(v.box.mr):.3f}')
    if map50 > best_map:
        best_map = map50
        best_conf = conf

print(f'\nBaseline (v2-retrain) : 0.5522')
print(f'Standard mAP@50      : {std_map50:.4f}  ({std_map50-0.5522:+.4f})')
print(f'TTA mAP@50           : {tta_map50:.4f}  ({tta_map50-0.5522:+.4f})')
print(f'Best conf={best_conf:.2f}          : {best_map:.4f}  ({best_map-0.5522:+.4f})')

=== INFERENCE COMPARISON ===
Standard inference...
Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 113 layers, 68,124,531 parameters, 0 gradients, 257.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 18.9±13.8 MB/s, size: 85.4 KB)
val: Scanning /content/checkpoints/detector_v3/test/labels... 92 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 92/92 179.1it/s 0.5s
val: New cache created: /content/checkpoints/detector_v3/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 1.5s/it 9.2s
                   all         92        147      0.508      0.503      0.463      0.222
Speed: 12.3ms preprocess, 68.7ms inference, 0.0ms loss, 1.5ms postprocess per image
Results saved to /content/runs/detect/val-4
Standard mAP@50 : 0.4628  P=0.508  R=0.503
TTA inference (augment=True)...
Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4

## SAHI Evaluation (Sliced Inference)

Slices each image into 640px tiles with 20% overlap, runs detection per tile,
then merges with NMS. Improves recall on thin/small cracks missed at full scale.

Evaluated on a 200-image test sample for speed.

In [20]:
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

sahi_model = AutoDetectionModel.from_pretrained(
    model_type='ultralytics',
    model_path=str(best_ckpt),
    confidence_threshold=best_conf,
    device=device,
)

test_images = sorted((TEST_DIR / 'images').glob('*.*'))
label_dir   = TEST_DIR / 'labels'

random.seed(42)
sahi_sample = random.sample(test_images, min(200, len(test_images)))
print(f'SAHI evaluation on {len(sahi_sample)} test images...')

def parse_yolo_gt(label_path, img_w, img_h):
    boxes = []
    if not Path(label_path).exists():
        return boxes
    for line in Path(label_path).read_text().strip().splitlines():
        parts = line.strip().split()
        if len(parts) != 5:
            continue
        _, cx, cy, bw, bh = map(float, parts)
        x1 = (cx - bw / 2) * img_w
        y1 = (cy - bh / 2) * img_h
        x2 = (cx + bw / 2) * img_w
        y2 = (cy + bh / 2) * img_h
        boxes.append([x1, y1, x2, y2])
    return boxes

def box_iou(b1, b2):
    ix1 = max(b1[0], b2[0]);  iy1 = max(b1[1], b2[1])
    ix2 = min(b1[2], b2[2]);  iy2 = min(b1[3], b2[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    a1 = (b1[2] - b1[0]) * (b1[3] - b1[1])
    a2 = (b2[2] - b2[0]) * (b2[3] - b2[1])
    return inter / (a1 + a2 - inter + 1e-7)

tp = fp = fn = 0
for img_path in tqdm(sahi_sample, desc='SAHI'):
    img = cv2.imread(str(img_path))
    if img is None:
        continue
    h, w = img.shape[:2]

    result = get_sliced_prediction(
        str(img_path), sahi_model,
        slice_height=640, slice_width=640,
        overlap_height_ratio=0.2, overlap_width_ratio=0.2,
        verbose=0,
    )

    pred_boxes = [
        [o.bbox.minx, o.bbox.miny, o.bbox.maxx, o.bbox.maxy]
        for o in result.object_prediction_list
    ]
    gt_boxes = parse_yolo_gt(label_dir / (img_path.stem + '.txt'), w, h)

    matched_gt = set()
    for pb in pred_boxes:
        best_iou, best_idx = 0.0, -1
        for i, gb in enumerate(gt_boxes):
            if i in matched_gt:
                continue
            v = box_iou(pb, gb)
            if v > best_iou:
                best_iou, best_idx = v, i
        if best_iou >= 0.5:
            tp += 1
            matched_gt.add(best_idx)
        else:
            fp += 1
    fn += len(gt_boxes) - len(matched_gt)

sahi_prec   = tp / (tp + fp + 1e-7)
sahi_recall = tp / (tp + fn + 1e-7)
sahi_f1     = 2 * sahi_prec * sahi_recall / (sahi_prec + sahi_recall + 1e-7)

print(f'\n=== SAHI Results (200-img sample, IoU>=0.5) ===')
print(f'TP={tp}  FP={fp}  FN={fn}')
print(f'Precision : {sahi_prec:.4f}')
print(f'Recall    : {sahi_recall:.4f}')
print(f'F1        : {sahi_f1:.4f}')

SAHI evaluation on 92 test images...


SAHI:   0%|          | 0/92 [00:00<?, ?it/s]


=== SAHI Results (200-img sample, IoU>=0.5) ===
TP=56  FP=122  FN=91
Precision : 0.3146
Recall    : 0.3810
F1        : 0.3446


## Final Summary & Save All to Drive

In [18]:
print('=' * 60)
print('DETECTION v3 — FINAL SUMMARY')
print('=' * 60)
print(f'Model    : YOLOv8x')
print(f'Training : 20ep@320px → 80ep@640px → 30ep@1024px')
print()
print(f'Validation mAP@50:')
print(f'  Phase 1 (320px)  : {hist["p1_map"]:.4f}')
print(f'  Phase 2 (640px)  : {hist["p2_map"]:.4f}')
print(f'  Phase 3 (1024px) : {hist["p3_map"]:.4f}')
print()
print(f'Test mAP@50:')
print(f'  Baseline (v2-retrain) : 0.5522')
print(f'  Standard              : {std_map50:.4f}  ({std_map50-0.5522:+.4f})')
print(f'  TTA (augment=True)    : {tta_map50:.4f}  ({tta_map50-0.5522:+.4f})')
print(f'  Best (conf={best_conf:.2f})       : {best_map:.4f}  ({best_map-0.5522:+.4f})')
print()
print(f'SAHI (200-img sample):')
print(f'  Precision : {sahi_prec:.4f}')
print(f'  Recall    : {sahi_recall:.4f}')
print(f'  F1        : {sahi_f1:.4f}')
print()
print(f'Best checkpoint : {best_ckpt}')
print(f'Drive backup    : {DRIVE_CKPT}')
print('=' * 60)

# Save metrics JSON
final_metrics = {
    'model': 'YOLOv8x',
    'phases': {
        'p1_val_map50': hist['p1_map'],
        'p2_val_map50': hist['p2_map'],
        'p3_val_map50': hist['p3_map'],
    },
    'test': {
        'standard_map50': std_map50,
        'tta_map50':       tta_map50,
        'best_conf':       best_conf,
        'best_map50':      best_map,
    },
    'sahi': {
        'sample_size': len(sahi_sample),
        'precision':   sahi_prec,
        'recall':      sahi_recall,
        'f1':          sahi_f1,
    },
    'baseline_v2_retrain': 0.5522,
    'timestamp': datetime.now().isoformat(),
}

metrics_path = CHECKPOINT_DIR / 'metrics_v3.json'
json.dump(final_metrics, open(metrics_path, 'w'), indent=2)
backup_to_drive(metrics_path, 'metrics_v3.json')

# Save best model to Drive root (for webapp / report)
backup_to_drive(best_ckpt, 'best_v3.pt')

print('All results saved to Drive.')

DETECTION v3 — FINAL SUMMARY
Model    : YOLOv8x
Training : 20ep@320px → 80ep@640px → 30ep@1024px

Validation mAP@50:
  Phase 1 (320px)  : 0.4242
  Phase 2 (640px)  : 0.5073
  Phase 3 (1024px) : 0.4882

Test mAP@50:
  Baseline (v2-retrain) : 0.5522
  Standard              : 0.4628  (-0.0894)
  TTA (augment=True)    : 0.4844  (-0.0678)
  Best (conf=0.15)       : 0.4351  (-0.1171)

SAHI (200-img sample):
  Precision : 0.3146
  Recall    : 0.3810
  F1        : 0.3446

Best checkpoint : /content/checkpoints/detector_v3/phase2_640px/weights/best.pt
Drive backup    : /content/drive/MyDrive/HeritagePreservation/checkpoints/detector_v3
  Drive backup: metrics_v3.json (0.0 MB)
  Drive backup: best_v3.pt (136.7 MB)
All results saved to Drive.
